In [ ]:
# importing necessary libraries
import csv
import json
import pandas as pd
import glob

In [ ]:
# Function to combine multiple CSV files into one CSV file
def combine_csv_files(file_list, output_file):
    combined_data = []
    for file in file_list:
        df = pd.read_csv(file)
        combined_data.append(df)
    combined_df = pd.concat(combined_data, ignore_index=True)
    combined_df.to_csv(output_file, index=False)
    return combined_df

In [ ]:
# Function to flatten a nested JSON object
def flatten_json(y, prefix=''):
    out = {}
    if isinstance(y, dict):
        for k, v in y.items():
            new_key = f"{prefix}_{k}" if prefix else k
            out.update(flatten_json(v, new_key))
    elif isinstance(y, list):
        for i, item in enumerate(y):
            new_key = f"{prefix}[{i}]"
            out.update(flatten_json(item, new_key))
    else:
        out[prefix] = y
    return out

In [ ]:
# Function to convert CSV to flat JSON
def csv_to_flat_json(csv_file_path, json_file_path):
    with open(csv_file_path, mode='r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        result = []
        for row in reader:
            parsed_row = {}
            for key, value in row.items():
                value = value.strip()
                if (value.startswith('{') and value.endswith('}')) or (value.startswith('[') and value.endswith(']')):
                    try:
                        parsed_row[key] = json.loads(value)
                    except json.JSONDecodeError:
                        parsed_row[key] = value
                else:
                    parsed_row[key] = value
            flat_row = flatten_json(parsed_row)
            result.append(flat_row)
    with open(json_file_path, 'w', encoding='utf-8') as jsonfile:
        json.dump(result, jsonfile, indent=4)
    return pd.DataFrame(result)

In [ ]:
# Combine CSV files and convert to flat JSON
csv_files = glob.glob(r"C:\Users\00zso\Downloads\Kickstarter_2025-08-12T06_24_51_926Z\Kickstarter*.csv")
combined_csv = 'kickstarter_combined.csv'
combine_csv_files(csv_files, combined_csv)
csv_to_flat_json(combined_csv, 'kickstarter_combined_flat.json')

In [ ]:
# Process the flat JSON to check for video existence and save as new JSON and CSV
full = pd.read_json('kickstarter_combined_flat.json')
full['video'] = full['video'].apply(
    lambda x: 'video exists' if pd.notnull(x) and '"status":"successful"' in str(x) else 'no video'
)
full.to_json('kickstarter_combined_clean.json', orient='records', indent=4)
full.to_csv('kickstarter_combined_clean.csv', index=False)

In [ ]:
# Function to check if certain fields are faulty (i.e., contain unparseable JSON)
def is_faulty(row):
    nested_fields = ['creator', 'profile', 'location']
    for field in nested_fields:
        if field in row and isinstance(row[field], str):
            val = row[field].strip()
            if (val.startswith('{') and val.endswith('}')) or (val.startswith('[') and val.endswith(']')):
                try:
                    json.loads(val)
                except json.JSONDecodeError:
                    return True
    return False

In [ ]:
# Read the cleaned JSON, identify faulty entries, and save them separately
df = pd.read_json('kickstarter_combined_clean.json')
df['faulty'] = df.apply(is_faulty, axis=1)

faulty_entries = df[df['faulty']].drop(columns=['faulty'])
faulty_entries.to_json('faulty_entries_from_combined_clean.json', orient='records', indent=4)
faulty_entries.to_csv('faulty_entries_from_combined_clean.csv', index=False)

clean_df = df[~df['faulty']].drop(columns=['faulty'])
clean_df.to_json('kickstarter_combined_clean_no_faulty.json', orient='records', indent=4)
clean_df.to_csv('kickstarter_combined_clean_no_faulty.csv', index=False)

In [ ]:
# Print shapes of the datasets for verification
print("Combined CSV shape:", pd.read_csv(combined_csv).shape)
print("Flat JSON loaded, shape:", full.shape)
print("Cleaned DataFrame shape:", clean_df.shape)